In [110]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from data_preparation.data_processor import DataProcessor

# XGBoost Implementation to Forecast Daily Waste Quantities

- Non-Autoregressive (NAR) Model, utilizing calendaric features only
- Autoregressive (AR) Model, utilizing additional lagged features 
- Autoregressive-Ratio (ARR) Model, utilizing additional lagged ratio features

## NAR Model

In [111]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = []

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= False,lagged_ratios= False ,fourier_terms= False, interaction_terms= False)

### Training and Predicting

In [112]:
yaml_file = "xgb_hyperparameters\\best_params_NoLags_20250501_1320.yaml"  

with open(yaml_file, 'r') as f:
    best_params_all = yaml.safe_load(f)

# Calculate split indices
total_samples = waste_dfs["Municipal"].shape[0]
train_end = int(total_samples * 0.8)  # First 80% for training
val1_end = int(total_samples * 0.9)   # Next 10% for validation set 1
# Last 10% remains for validation set 2

waste_quantity_preds_NoLags = {}
waste_quantity_preds_val1 = {}  # New dict for first validation set
waste_quantity_preds_val2 = {}  # New dict for second validation set

for waste, df in waste_dfs.items():
    X = df.drop(columns=['quantity_tons'])
    y = df['quantity_tons']

    # Split the data
    X_train = X[:train_end]
    y_train = y[:train_end]
    
    X_val1 = X[train_end:val1_end]
    y_val1 = y[train_end:val1_end]
    
    X_train_2 = X[:val1_end]
    y_train_2 = y[:val1_end]

    X_val2 = X[val1_end:]
    y_val2 = y[val1_end:]
    
    best_params = best_params_all[waste]
    
    # Create and train the model with best parameters
    best_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        **best_params
    )
    
    # Make predictions on both validation sets
    best_model.fit(X_train, y_train)
    y_pred_val1 = best_model.predict(X_val1)

    best_model.fit(X_train_2, y_train_2)
    y_pred_val2 = best_model.predict(X_val2)
    
    # Store predictions
    waste_quantity_preds_val1[waste] = y_pred_val1
    waste_quantity_preds_val2[waste] = y_pred_val2

    # Plot feature importance
    #plt.figure(figsize=(10, 8))
    #xgb.plot_importance(best_model, max_num_features=20, height=0.8, importance_type="gain")
    #plt.title(f"Feature Importance for {waste}")
    #plt.show()

### Calculating Error Metrics

In [113]:
print("Test Set 1 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val1 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[train_end:val1_end] for waste in waste_dfs],
    axis=0
)
total_predicted_val1 = np.sum(list(waste_quantity_preds_val1.values()), axis=0)

# Individual waste type evaluation
for waste in waste_dfs:
    actual_val1 = waste_dfs[waste]['quantity_tons'].values[train_end:val1_end]
    predicted_val1 = waste_quantity_preds_val1[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val1, color="blue", label="Actual")
    #plt.plot(predicted_val1, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 1: Actual vs Predicted for {waste}")
    #plt.legend()
    
    
    print(f"\n{waste}:")
    print(f"RMSE: {root_mean_squared_error(actual_val1, predicted_val1):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val1, predicted_val1):.2f}")
#plt.show()



print("\n" + "="*50)
print("Test Set 2 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val2 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[val1_end:] for waste in waste_dfs],
    axis=0
)
total_predicted_val2 = np.sum(list(waste_quantity_preds_val2.values()), axis=0)

# Individual waste type evaluation
for waste in waste_dfs:
    actual_val2 = waste_dfs[waste]['quantity_tons'].values[val1_end:]
    predicted_val2 = waste_quantity_preds_val2[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val2, color="blue", label="Actual")
    #plt.plot(predicted_val2, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 2: Actual vs Predicted for {waste}")
    #plt.legend()
    
    
    print(f"\n{waste}:")
    print(f"RMSE: {root_mean_squared_error(actual_val2, predicted_val2):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val2, predicted_val2):.2f}")
#plt.show()

Test Set 1 Evaluation

Municipal:
RMSE: 49.90
MAE: 38.94

Industrial:
RMSE: 41.23
MAE: 34.32

Organic:
RMSE: 43.07
MAE: 32.39

Construction:
RMSE: 37.45
MAE: 29.87

Commercial:
RMSE: 28.54
MAE: 22.79

Test Set 2 Evaluation

Municipal:
RMSE: 51.07
MAE: 37.65

Industrial:
RMSE: 42.12
MAE: 31.63

Organic:
RMSE: 28.38
MAE: 21.20

Construction:
RMSE: 21.21
MAE: 16.82

Commercial:
RMSE: 38.07
MAE: 29.23


## AR Model

In [114]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = [6,7,13,14,20,21]

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= True,lagged_ratios= False ,fourier_terms= False, interaction_terms= False)

### Training and Predicting

In [115]:
yaml_file = "xgb_hyperparameters\\best_params_Lags_20250501_1322.yaml"  

with open(yaml_file, 'r') as f:
    best_params_all = yaml.safe_load(f)


waste_quantity_preds_Lags_val1 = {}  
waste_quantity_preds_Lags_val2 = {}  

for waste, df in waste_dfs.items():
    X = df.drop(columns=['quantity_tons'])
    y = df['quantity_tons']

    # Split the data
    X_train = X[:train_end]
    y_train = y[:train_end]
    
    X_val1 = X[train_end:val1_end]
    y_val1 = y[train_end:val1_end]

    X_train_2 = X[:val1_end]
    y_train_2 = y[:val1_end]
    
    X_val2 = X[val1_end:]
    y_val2 = y[val1_end:]
    
    best_params = best_params_all[waste]
    
    # Create and train the model with best parameters
    best_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        **best_params
    )

    # Make predictions on both validation sets
    best_model.fit(X_train, y_train)
    y_pred_val1 = best_model.predict(X_val1)

    best_model.fit(X_train_2, y_train_2)
    y_pred_val2 = best_model.predict(X_val2)
    
    # Store predictions
    waste_quantity_preds_Lags_val1[waste] = y_pred_val1
    waste_quantity_preds_Lags_val2[waste] = y_pred_val2

    # Plot feature importance
    #plt.figure(figsize=(10, 8))
    #xgb.plot_importance(best_model, max_num_features=20, height=0.8, importance_type="gain")
    #plt.title(f"Feature Importance for {waste}")
    #plt.show()

In [116]:
print("Test Set 1 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val1 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[train_end:val1_end] for waste in waste_dfs],
    axis=0
)
total_predicted_val1 = np.sum(list(waste_quantity_preds_Lags_val1.values()), axis=0)


# Individual waste type evaluation
for waste in waste_dfs:
    actual_val1 = waste_dfs[waste]['quantity_tons'].values[train_end:val1_end]
    predicted_val1 = waste_quantity_preds_Lags_val1[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val1, color="blue", label="Actual")
    #plt.plot(predicted_val1, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 1: Actual vs Predicted for {waste}")
    #plt.legend()
    
    
    print(f"\n{waste}")
    print(f"RMSE: {root_mean_squared_error(actual_val1, predicted_val1):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val1, predicted_val1):.2f}")
#plt.show()


print("\n" + "="*50)
print("Test Set 2 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val2 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[val1_end:] for waste in waste_dfs],
    axis=0
)
total_predicted_val2 = np.sum(list(waste_quantity_preds_Lags_val2.values()), axis=0)


# Individual waste type evaluation
for waste in waste_dfs:
    actual_val2 = waste_dfs[waste]['quantity_tons'].values[val1_end:]
    predicted_val2 = waste_quantity_preds_Lags_val2[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val2, color="blue", label="Actual")
    #plt.plot(predicted_val2, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 2: Actual vs Predicted for {waste}")
    #plt.legend()
    
    
    print(f"\n{waste}")
    print(f"RMSE: {root_mean_squared_error(actual_val2, predicted_val2):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val2, predicted_val2):.2f}")
#plt.show()


Test Set 1 Evaluation

Municipal
RMSE: 55.06
MAE: 41.20

Industrial
RMSE: 40.44
MAE: 33.24

Organic
RMSE: 44.79
MAE: 34.48

Construction
RMSE: 36.89
MAE: 29.12

Commercial
RMSE: 29.89
MAE: 24.13

Test Set 2 Evaluation

Municipal
RMSE: 49.54
MAE: 36.89

Industrial
RMSE: 43.81
MAE: 32.78

Organic
RMSE: 26.83
MAE: 19.12

Construction
RMSE: 21.43
MAE: 18.21

Commercial
RMSE: 40.72
MAE: 31.57


## ARR Model

Now instead of using lagged features, lagged ratios are employed. Given a time series $ \{X_t\} $, the lagged ratio $ R_{t,k} $ at time $ t $ is defined as:

$$
R_{t}^{k,m} = \frac{X_{t-k}}{X_{t-m}}
$$

where:
- $t$ is the current time step,
- $k$ and $m$ are the lags.

For example at time $t=6$, the lagged ratio $R_{6}^{3,4}$ is calculated as $\frac{X_3}{X_2}$.

In [117]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}

lags = [6,7,13,14,20,21]

for waste in unique_waste:
    prep_data_company = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)
    waste_dfs[waste] = fetcher.create_xgboost_features(prep_data_company,waste_type= waste ,lags=lags, lagged_features= False,lagged_ratios= True ,fourier_terms= False, interaction_terms= False)

In [118]:
yaml_file = "xgb_hyperparameters\\best_params_LagRatios_20250501_1324.yaml"  

with open(yaml_file, 'r') as f:
    best_params_all = yaml.safe_load(f)


waste_quantity_preds_LR_val1 = {}  
waste_quantity_preds_LR_val2 = {}  

for waste, df in waste_dfs.items():
    X = df.drop(columns=['quantity_tons'])
    y = df['quantity_tons']

    # Split the data
    X_train = X[:train_end]
    y_train = y[:train_end]
    
    X_val1 = X[train_end:val1_end]
    y_val1 = y[train_end:val1_end]

    X_train_2 = X[:val1_end]
    y_train_2 = y[:val1_end]
    
    X_val2 = X[val1_end:]
    y_val2 = y[val1_end:]
    
    best_params = best_params_all[waste]
    
    # Create and train the model with best parameters
    best_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        **best_params
    )

    # Make predictions on both validation sets
    best_model.fit(X_train, y_train)
    y_pred_val1 = best_model.predict(X_val1)

    best_model.fit(X_train_2, y_train_2)
    y_pred_val2 = best_model.predict(X_val2)
    
    # Store predictions
    waste_quantity_preds_LR_val1[waste] = y_pred_val1
    waste_quantity_preds_LR_val2[waste] = y_pred_val2

    # Plot feature importance
    #plt.figure(figsize=(10, 8))
    #xgb.plot_importance(best_model, max_num_features=20, height=0.8, importance_type="gain")
    #plt.title(f"Feature Importance for {waste}")
    #plt.show()

In [119]:
print("Test Set 1 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val1 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[train_end:val1_end] for waste in waste_dfs],
    axis=0
)
total_predicted_val1 = np.sum(list(waste_quantity_preds_LR_val1.values()), axis=0)


# Individual waste type evaluation
for waste in waste_dfs:
    actual_val1 = waste_dfs[waste]['quantity_tons'].values[train_end:val1_end]
    predicted_val1 = waste_quantity_preds_LR_val1[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val1, color="blue", label="Actual")
    #plt.plot(predicted_val1, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 1: Actual vs Predicted for {waste}")
    #plt.legend()
    
    print(f"\n{waste}")
    print(f"RMSE: {root_mean_squared_error(actual_val1, predicted_val1):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val1, predicted_val1):.2f}")
#plt.show()

print("\n" + "="*50)
print("Test Set 2 Evaluation")
print("="*50)

# Aggregate all waste types for total evaluation
total_actual_val2 = np.sum(
    [waste_dfs[waste]['quantity_tons'].values[val1_end:] for waste in waste_dfs],
    axis=0
)
total_predicted_val2 = np.sum(list(waste_quantity_preds_LR_val2.values()), axis=0)


# Individual waste type evaluation
for waste in waste_dfs:
    actual_val2 = waste_dfs[waste]['quantity_tons'].values[val1_end:]
    predicted_val2 = waste_quantity_preds_LR_val2[waste]
    
    #plt.figure(figsize=(16, 8))
    #plt.plot(actual_val2, color="blue", label="Actual")
    #plt.plot(predicted_val2, color="red", linestyle="dashed", label="Predicted")
    #plt.title(f"Validation Set 2: Actual vs Predicted for {waste}")
    #plt.legend()
    
    
    print(f"\n{waste}")
    print(f"RMSE: {root_mean_squared_error(actual_val2, predicted_val2):.2f}")
    print(f"MAE: {mean_absolute_error(actual_val2, predicted_val2):.2f}")
#plt.show()

Test Set 1 Evaluation

Municipal
RMSE: 49.96
MAE: 38.23

Industrial
RMSE: 41.10
MAE: 34.06

Organic
RMSE: 46.09
MAE: 35.15

Construction
RMSE: 37.47
MAE: 29.31

Commercial
RMSE: 28.74
MAE: 22.77

Test Set 2 Evaluation

Municipal
RMSE: 50.99
MAE: 37.88

Industrial
RMSE: 42.52
MAE: 32.05

Organic
RMSE: 26.89
MAE: 20.35

Construction
RMSE: 22.20
MAE: 16.86

Commercial
RMSE: 39.02
MAE: 30.03


## Holt-Winters Baseline

In [120]:
waste_data = pd.read_csv("../synthetic_waste_data.csv")

fetcher = DataProcessor(waste_data)

unique_waste = fetcher.waste_data['waste_type'].unique()
waste_dfs = {}


for waste in unique_waste:
    waste_dfs[waste] = fetcher.agg_quantity(waste_type=waste, by_waste_type= True)

In [121]:
for waste in unique_waste:
    df = waste_dfs[waste].copy()
    df.index = pd.to_datetime(df.index)
    df = df.asfreq('D')  # Set daily frequency
    df = df.dropna()

    total_samples = len(df)
    train_end = int(total_samples * 0.8)
    val1_end = int(total_samples * 0.9)

    train = df['quantity_tons'][:train_end]
    train_2 = df['quantity_tons'][:val1_end]
    actual_val1 = df['quantity_tons'][train_end:val1_end]
    actual_val2 = df['quantity_tons'][val1_end:]

    # Holt-Winters: seasonal only, no trend
    model = ExponentialSmoothing(
        train,
        trend=None,
        seasonal='add',              # or 'mul' depending on your data
        seasonal_periods=365,          # yearly seasonality
        initialization_method="estimated"
    )
    model_fit = model.fit()

    forecast_val1 = model_fit.forecast(len(actual_val1))

    model = ExponentialSmoothing(
        train_2,
        trend=None,
        seasonal='add',              # or 'mul' depending on your data
        seasonal_periods=365,          # yearly seasonality
        initialization_method="estimated"
    )
    model_fit = model.fit()
    forecast_val2 = model_fit.forecast(len(actual_val2))


    rmse_val1 = root_mean_squared_error(actual_val1, forecast_val1)
    mae_val1 = mean_absolute_error(actual_val1, forecast_val1)

    rmse_val2 = root_mean_squared_error(actual_val2, forecast_val2)
    mae_val2 = mean_absolute_error(actual_val2, forecast_val2)

    print(f'\nHolt-Winters Forecast Evaluation for {waste}')
    print('-'*40)
    print(f'Validation Set 1 (First 10%): RMSE = {rmse_val1:.2f}, MAE = {mae_val1:.2f}')
    print(f'Validation Set 2 (Last 10%):  RMSE = {rmse_val2:.2f}, MAE = {mae_val2:.2f}')



Holt-Winters Forecast Evaluation for Municipal
----------------------------------------
Validation Set 1 (First 10%): RMSE = 92.31, MAE = 72.45
Validation Set 2 (Last 10%):  RMSE = 82.90, MAE = 63.47

Holt-Winters Forecast Evaluation for Industrial
----------------------------------------
Validation Set 1 (First 10%): RMSE = 51.09, MAE = 42.93
Validation Set 2 (Last 10%):  RMSE = 58.84, MAE = 45.01

Holt-Winters Forecast Evaluation for Organic
----------------------------------------
Validation Set 1 (First 10%): RMSE = 58.37, MAE = 44.94
Validation Set 2 (Last 10%):  RMSE = 43.40, MAE = 33.87

Holt-Winters Forecast Evaluation for Construction
----------------------------------------
Validation Set 1 (First 10%): RMSE = 46.97, MAE = 37.55
Validation Set 2 (Last 10%):  RMSE = 29.85, MAE = 24.74

Holt-Winters Forecast Evaluation for Commercial
----------------------------------------
Validation Set 1 (First 10%): RMSE = 44.61, MAE = 36.70
Validation Set 2 (Last 10%):  RMSE = 54.84, MAE 

# Diebold-Mariano Test for Test Set 1

I performed the [Diebold-Mariano test](https://pypi.org/project/dieboldmariano/) to statistically compare forecast accuracy between models. This test evaluates whether two forecasts have significantly different prediction errors by testing the null hypothesis of equal predictive accuracy.

In [122]:
waste_quantity_preds_prophet_df1 = pd.read_csv('prophet_predictions\\prophet_preds_1.csv')

waste_quantity_preds_prophet1 = waste_quantity_preds_prophet_df1.to_dict(orient='list')


waste_quantity_preds_prophet1 = {waste: np.array(preds) for waste, preds in waste_quantity_preds_prophet1.items()}

In [123]:
from dieboldmariano import dm_test
from statsmodels.tsa.stattools import adfuller
from itertools import combinations
import numpy as np

# Add Prophet to model dict
models = {
    "NAR": waste_quantity_preds_val1,
    "AR": waste_quantity_preds_Lags_val1,
    "ARR": waste_quantity_preds_LR_val1,
    "Prophet": waste_quantity_preds_prophet1
}

# Generate all pairwise combinations of models
model_pairs = list(combinations(models.keys(), 2))

results = {}

for waste_type in unique_waste:
    true_values = waste_dfs[waste_type]["quantity_tons"].values
    results[waste_type] = {}

    for model1, model2 in model_pairs:
        
        preds1 = models[model1][waste_type]
        preds2 = models[model2][waste_type]

        # Ensure same length
        min_len = min(len(true_values), len(preds1), len(preds2))
        T = true_values[-min_len:]
        P1 = preds1[-min_len:]
        P2 = preds2[-min_len:]

        # Perform DM test (squared error loss)
        dm_stat, p_val = dm_test(T, P1, P2, loss=lambda u, v: (u - v) ** 2)

        # Compute loss differential time series
        d_series = np.array([(t - p1) ** 2 - (t - p2) ** 2 for t, p1, p2 in zip(T, P1, P2)])

        # ADF test
        adf_stat, adf_p, _, _, crit_vals, _ = adfuller(d_series, autolag='BIC')
        # Store results
        results[waste_type][f"{model1} vs {model2}"] = {
            "DM_stat": dm_stat,
            "p_value": p_val,
            "ADF_stat": adf_stat,
            "ADF_p": adf_p
        }

# Print summary
for waste_type, comparisons in results.items():
    print(f"\n=== {waste_type.upper():^20} ===")
    print("-"*50)
    print(f"{'Comparison':<25} | {'DM Stat':>8} | {'p-value':>8} | {'ADF Stat':>8} | {'ADF p':>8}")
    print("-"*50)
    for comp, outcome in comparisons.items():
        print(f"{comp:<25} | {outcome['DM_stat']:>8.3f} | {outcome['p_value']:>8.3f} | "
              f"{outcome['ADF_stat']:>8.3f} | {outcome['ADF_p']:>8.3f}")


===      MUNICIPAL       ===
--------------------------------------------------
Comparison                |  DM Stat |  p-value | ADF Stat |    ADF p
--------------------------------------------------
NAR vs AR                 |    0.497 |    0.620 |   -3.434 |    0.010
NAR vs ARR                |    0.149 |    0.882 |   -8.413 |    0.000
NAR vs Prophet            |    0.603 |    0.547 |   -6.511 |    0.000
AR vs ARR                 |   -0.433 |    0.666 |   -3.344 |    0.013
AR vs Prophet             |   -0.168 |    0.867 |   -1.364 |    0.599
ARR vs Prophet            |    0.545 |    0.587 |   -7.565 |    0.000

===      INDUSTRIAL      ===
--------------------------------------------------
Comparison                |  DM Stat |  p-value | ADF Stat |    ADF p
--------------------------------------------------
NAR vs AR                 |   -0.158 |    0.874 |  -11.706 |    0.000
NAR vs ARR                |    2.243 |    0.027 |  -10.174 |    0.000
NAR vs Prophet            |   -1.284

# Diebold-Mariano Test for Test Set 2

In [124]:
waste_quantity_preds_prophet_df2 = pd.read_csv('prophet_predictions\\prophet_preds_2.csv')

waste_quantity_preds_prophet2 = waste_quantity_preds_prophet_df2.to_dict(orient='list')


waste_quantity_preds_prophet2 = {waste: np.array(preds) for waste, preds in waste_quantity_preds_prophet2.items()}

In [125]:
# Add Prophet to model dict
models = {
    "NAR": waste_quantity_preds_val2,
    "AR": waste_quantity_preds_Lags_val2,
    "ARR": waste_quantity_preds_LR_val2,
    "Prophet": waste_quantity_preds_prophet2
}

# Generate all pairwise combinations of models
model_pairs = list(combinations(models.keys(), 2))

results = {}

for waste_type in unique_waste:
    true_values = waste_dfs[waste_type]["quantity_tons"].values
    results[waste_type] = {}

    for model1, model2 in model_pairs:
        
        preds1 = models[model1][waste_type]
        preds2 = models[model2][waste_type]

        # Ensure same length
        min_len = min(len(true_values), len(preds1), len(preds2))
        T = true_values[-min_len:]
        P1 = preds1[-min_len:]
        P2 = preds2[-min_len:]

        # Perform DM test (squared error loss)
        dm_stat, p_val = dm_test(T, P1, P2, loss=lambda u, v: (u - v) ** 2)

        # Compute loss differential time series
        d_series = np.array([(t - p1) ** 2 - (t - p2) ** 2 for t, p1, p2 in zip(T, P1, P2)])

        # ADF test
        adf_stat, adf_p, _, _, crit_vals, _ = adfuller(d_series, autolag='BIC')
        # Store results
        results[waste_type][f"{model1} vs {model2}"] = {
            "DM_stat": dm_stat,
            "p_value": p_val,
            "ADF_stat": adf_stat,
            "ADF_p": adf_p
        }

# Print summary
for waste_type, comparisons in results.items():
    print(f"\n=== {waste_type.upper():^20} ===")
    print("-"*50)
    print(f"{'Comparison':<25} | {'DM Stat':>8} | {'p-value':>8} | {'ADF Stat':>8} | {'ADF p':>8}")
    print("-"*50)
    for comp, outcome in comparisons.items():
        print(f"{comp:<25} | {outcome['DM_stat']:>8.3f} | {outcome['p_value']:>8.3f} | "
              f"{outcome['ADF_stat']:>8.3f} | {outcome['ADF_p']:>8.3f}")


===      MUNICIPAL       ===
--------------------------------------------------
Comparison                |  DM Stat |  p-value | ADF Stat |    ADF p
--------------------------------------------------
NAR vs AR                 |   -0.632 |    0.529 |   -8.903 |    0.000
NAR vs ARR                |    0.108 |    0.914 |   -9.714 |    0.000
NAR vs Prophet            |    1.776 |    0.078 |   -9.026 |    0.000
AR vs ARR                 |    0.924 |    0.358 |   -7.962 |    0.000
AR vs Prophet             |    1.393 |    0.167 |   -7.883 |    0.000
ARR vs Prophet            |    1.842 |    0.068 |   -9.243 |    0.000

===      INDUSTRIAL      ===
--------------------------------------------------
Comparison                |  DM Stat |  p-value | ADF Stat |    ADF p
--------------------------------------------------
NAR vs AR                 |   -0.541 |    0.590 |   -8.927 |    0.000
NAR vs ARR                |   -0.287 |    0.775 |  -10.697 |    0.000
NAR vs Prophet            |    1.300